# XGBoost Post-Mid Model Training
Train XGBoost models on NYISO + Mesonet fusion data with lag feature grid search

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import json
from pathlib import Path

print("XGBoost version:", xgb.__version__)

In [ ]:
import sys
if sys.platform == 'win32':
    import asyncio
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
import gc
gc.collect()

In [ ]:
# Configuration
CONFIG = {
    "train_start": 2001,
    "train_end": 2021,
    "val_year": 2022,
    "test_years": [2023, 2024, 2025],

    # Fixed XGBoost model parameters
    "model": {
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "tree_method": "hist",
        "early_stopping_rounds": 20,
        "random_state": 42
    }
}

# Lag feature grid (only parameter being searched)
LAG_GRID = {
    'five': [
        [1, 5, 15],          # short-term only
        [1, 5, 15, 60],      # add 5 hours
        [1, 12, 36, 72]      # 1h, 3h, 6h
    ],
    'quarter': [
        [1, 7, 30],          
        [1, 3, 7, 30],       
        [1, 7, 30, 90]       
    ],
    'hourly': [
        [1, 24, 168],        # 1 hour, day, week
        [1, 24, 72, 168],    # 3 days
        [1, 24, 168, 336]    # 2 weeks 
    ],
    'daily': [
        [1, 7, 30],          # 1 day, week, month
        [1, 3, 7, 30],       # 3 days
        [1, 7, 30, 90]       # 3 months
    ]
}

print("Configuration loaded")
print(f"Model params: {CONFIG['model']}")


## Load Master Parquet Data

In [ ]:
# Path to master parquet (fusion of NYISO + Mesonet)
MASTER_PATH = Path(r"c:/Users/Matt/Desktop/CS506/CS506_Project/1_LIB/master/master.parquet")

print(f"Loading fusion dataset from: {MASTER_PATH}")
df = pd.read_parquet(MASTER_PATH)

# Rename and process datetime
df.rename(columns={"datetime": "Time_Stamp"}, inplace=True)
df["Time_Stamp"] = pd.to_datetime(df["Time_Stamp"], utc=True)
df = df.sort_values("Time_Stamp").set_index("Time_Stamp")

# Clean column names (remove special characters)
df.columns = (
    df.columns.str.replace(r"[\[\]\(\)/ ]", "_", regex=True)
              .str.replace(r"__+", "_", regex=True)
              .str.strip("_")
)

print(f"Loaded fusion dataset: {len(df):,} rows")
print(f"Columns: {df.columns.tolist()}")
print(f"\nDate range: {df.index.min()} to {df.index.max()}")


In [ ]:
# Split weather and load columns
weather_cols = [c for c in df.columns if c not in ["Load", "PTID"]]

# Fill NAs safely (forward fill → 0)
df[weather_cols] = df[weather_cols].ffill().fillna(0)

print(f"\nWeather features ({len(weather_cols)}): {weather_cols[:10]}...")
print(f"\nFirst few rows:")
print(df.head())


In [ ]:
# Check time resolution
time_diffs = df.index.to_series().diff()
print("\nTime difference between consecutive rows:")
print(time_diffs.value_counts().head(10))


## Create Time Aggregations

In [ ]:
print("Creating aggregates...")

# 5-minute (original resolution from master parquet)
five = df.copy()

# 15-minute
quarter = df.resample("15min").agg({
    **{c: "mean" for c in weather_cols},
    "Load": "mean"
})

# Hourly
hourly = df.resample("1h").agg({
    **{c: "mean" for c in weather_cols},
    "Load": "mean"
})

# Daily
daily = df.resample("1d").agg({
    **{c: "mean" for c in weather_cols},
    "Load": "mean"
})

AGG_DFS = {
    'five': five,
    'quarter': quarter,
    'hourly': hourly,
    'daily': daily
}

# Print shapes
for name, agg_df in AGG_DFS.items():
    print(f"{name:8} → {len(agg_df):,} rows")


In [ ]:
# Display sample of each aggregation
print("\n5-minute sample:")
print(five.head(3))
print("\nQuarter-hour sample:")
print(quarter.head(3))


## XGBoost Training Function

In [ ]:
def run_xgboost(df, lags, agg_name, plot=False):
    """
    Train XGBoost with fixed hyperparameters.
    Only lag features are varied.
    """
    start_time = time.time()

    df = df.copy().reset_index()
    df["year"] = df["Time_Stamp"].dt.year

    # Create lag features
    for lag in lags:
        df[f"lag_{lag}"] = df["Load"].shift(lag)

    df = df.dropna()

    # Identify feature columns
    weather_cols_clean = [
        c for c in df.columns
        if c not in ["Time_Stamp", "Load", "PTID", "year"]
        and not c.startswith("lag_")
    ]
    
    lag_cols = [f"lag_{l}" for l in lags]
    feature_cols = weather_cols_clean + lag_cols

    # Split data
    train = df[
        (df["year"] >= CONFIG["train_start"]) &
        (df["year"] <= CONFIG["train_end"])
    ]
    val = df[df["year"] == CONFIG["val_year"]]
    test = df[df["year"].isin(CONFIG["test_years"])]

    X_train = train[feature_cols]
    y_train = train["Load"]
    X_val = val[feature_cols]
    y_val = val["Load"]
    X_test = test[feature_cols]
    y_test = test["Load"]

    print(f"\n[{agg_name}] Train={len(train):,}, Val={len(val):,}, Test={len(test):,}")
    print(f"  Features: {len(feature_cols)} ({len(weather_cols_clean)} weather + {len(lag_cols)} lags)")
    print(f"  Lags={lags}")

    # Train model
    model = xgb.XGBRegressor(**CONFIG["model"])
    model.fit(
        X_train, y_train, 
        eval_set=[(X_val, y_val)], 
        verbose=False
    )

    # Predictions
    y_val_pred = model.predict(X_val)
    y_pred = model.predict(X_test)

    # Validation metrics
    rmse_val = root_mean_squared_error(y_val, y_val_pred)
    mae_val = mean_absolute_error(y_val, y_val_pred)
    r2_val = r2_score(y_val, y_val_pred)
    mask_val = y_val != 0
    mape_val = (abs((y_val[mask_val] - y_val_pred[mask_val]) / y_val[mask_val])).mean() * 100

    # Test metrics
    rmse_test = root_mean_squared_error(y_test, y_pred)
    mae_test = mean_absolute_error(y_test, y_pred)
    r2_test = r2_score(y_test, y_pred)
    mask_test = y_test != 0
    mape_test = (abs((y_test[mask_test] - y_pred[mask_test]) / y_test[mask_test])).mean() * 100

    elapsed = time.time() - start_time

    print(f"  VAL  → MAPE={mape_val:.3f}% | MAE={mae_val:.2f} | RMSE={rmse_val:.2f} | R²={r2_val:.4f}")
    print(f"  TEST → MAPE={mape_test:.3f}% | MAE={mae_test:.2f} | RMSE={rmse_test:.2f} | R²={r2_test:.4f}")
    print(f"  Time → {elapsed:.2f}s")

    if plot:
        plt.figure(figsize=(14, 6))
        plt.plot(test["Time_Stamp"].values, y_test.values, label="Actual", alpha=0.6)
        plt.plot(test["Time_Stamp"].values, y_pred, label="Predicted", alpha=0.8)
        plt.title(f"{agg_name} — Actual vs Predicted Load (lags={lags})")
        plt.xlabel("Time")
        plt.ylabel("Load (MW)")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return {
        "MAPE_val": mape_val,
        "MAE_val": mae_val,
        "RMSE_val": rmse_val,
        "R2_val": r2_val,
        "MAPE": mape_test,
        "MAE": mae_test,
        "RMSE": rmse_test,
        "R2": r2_test,
        "Time_s": elapsed,
        "lags": lags,
        "params": CONFIG["model"].copy()
    }

print("XGBoost training function defined")


## Grid Search for Optimal Lags

In [ ]:
def gridsearch_for_agg(df, agg_name):
    """
    Grid search over lag configurations.
    Select best based on validation MAPE.
    """
    lag_candidates = LAG_GRID[agg_name]
    print(f"\n{'='*60}")
    print(f"GRID SEARCH: {agg_name.upper()}")
    print(f"{'='*60}")
    print(f"Fixed hyperparameters: {CONFIG['model']}")
    print(f"Lag candidates: {lag_candidates}")

    best = None
    best_lags = None
    all_results = []

    for lags in lag_candidates:
        print(f"\n--- Trying lags={lags} ---")
        metrics = run_xgboost(df, lags, agg_name, plot=False)
        all_results.append(metrics)

        if (best is None) or (metrics["MAPE_val"] < best["MAPE_val"]):
            best = metrics
            best_lags = lags

    print(f"\n{'='*60}")
    print(f"BEST CONFIG for {agg_name}:")
    print(f"{'='*60}")
    print(f"  Lags:      {best_lags}")
    print(f"  VAL MAPE:  {best['MAPE_val']:.3f}%")
    print(f"  TEST MAPE: {best['MAPE']:.3f}%")
    print(f"  TEST MAE:  {best['MAE']:.2f}")
    print(f"  TEST RMSE: {best['RMSE']:.2f}")
    print(f"  TEST R²:   {best['R2']:.4f}")

    # Plot best configuration
    print("\nGenerating plot for best configuration...")
    _ = run_xgboost(df, best_lags, agg_name, plot=True)

    return best, all_results

print("Grid search function defined")


## Run Grid Search for All Aggregations

In [ ]:
total_start = time.time()

results = {}
all_results_by_agg = {}

for agg_name, df_agg in AGG_DFS.items():
    print(f"\n\n{'#'*70}")
    print(f"# Processing: {agg_name.upper()} aggregation")
    print(f"{'#'*70}")
    
    best_result, all_configs = gridsearch_for_agg(df_agg, agg_name)
    results[agg_name] = best_result
    all_results_by_agg[agg_name] = all_configs

total_time = time.time() - total_start
print(f"\n\nTotal runtime: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")


## Results Summary

In [ ]:
# Build summary table (TEST metrics only)
summary = {}
for agg_name, res in results.items():
    summary[agg_name] = {
        "MAPE": res["MAPE"],
        "MAE": res["MAE"],
        "RMSE": res["RMSE"],
        "R2": res["R2"],
        "Time_s": res["Time_s"],
    }

summary_df = pd.DataFrame(summary).T[["MAPE", "MAE", "RMSE", "R2", "Time_s"]]

print("\n" + "="*70)
print("FINAL RESULTS SUMMARY (TEST SET)")
print("="*70)
print(summary_df.round(4))


In [ ]:
# Display optimal lags
print("\n" + "="*70)
print("OPTIMAL LAG CONFIGURATIONS")
print("="*70)
for agg_name, res in results.items():
    print(f"{agg_name:8} → {res['lags']}")


In [ ]:
# Save results to JSON
output_file = "results_new.json"
with open(output_file, "w") as f:
    json.dump(summary, f, indent=4)

print(f"\n✓ Saved results to: {output_file}")

# Also save full results with all configurations
full_results_file = "results_new_full.json"
full_results = {
    "summary": summary,
    "best_configs": {k: v for k, v in results.items()},
    "all_configs": all_results_by_agg
}
with open(full_results_file, "w") as f:
    json.dump(full_results, f, indent=4)

print(f"✓ Saved full results to: {full_results_file}")


## Comparison with Baseline (if available)

In [ ]:
# Try to load old results for comparison
try:
    with open("results_old.json", "r") as f:
        old_results = json.load(f)
    
    print("\n" + "="*70)
    print("IMPROVEMENT OVER BASELINE (NYISO-only)")
    print("="*70)
    
    comparison = pd.DataFrame({
        "Old_MAPE": [old_results.get(k, {}).get("MAPE", np.nan) for k in summary.keys()],
        "New_MAPE": [summary[k]["MAPE"] for k in summary.keys()],
        "Old_R2": [old_results.get(k, {}).get("R2", np.nan) for k in summary.keys()],
        "New_R2": [summary[k]["R2"] for k in summary.keys()],
    }, index=summary.keys())
    
    comparison["MAPE_Δ%"] = ((comparison["Old_MAPE"] - comparison["New_MAPE"]) / comparison["Old_MAPE"] * 100)
    comparison["R2_Δ%"] = ((comparison["New_R2"] - comparison["Old_R2"]) / comparison["Old_R2"] * 100)
    
    print(comparison.round(4))
    
except FileNotFoundError:
    print("\nresults_old.json not found - skipping comparison")
